In [1]:
from datasets import load_dataset

dataset = load_dataset("PatronusAI/financebench")

/home/pranavamurthy-ks/dev/financebench-safety/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset

DatasetDict({
    train: Dataset({
        features: ['financebench_id', 'company', 'doc_name', 'question_type', 'question_reasoning', 'domain_question_num', 'question', 'answer', 'justification', 'dataset_subset_label', 'evidence', 'gics_sector', 'doc_type', 'doc_period', 'doc_link'],
        num_rows: 150
    })
})

In [3]:
example = dataset["train"][0]

question = example["question"]
expected_answer = example["answer"]
evidence = example["evidence"][0]["evidence_text"]

In [4]:
example["evidence"][0].keys()

dict_keys(['evidence_text', 'doc_name', 'evidence_page_num', 'evidence_text_full_page'])

In [5]:
print("QUESTION:", question)
print("EXPECTED ANSWER:", expected_answer)
print("EVIDENCE:", evidence[:500])


QUESTION: What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.
EXPECTED ANSWER: $1577.00
EVIDENCE: Table of Contents 
3M Company and Subsidiaries
Consolidated Statement of Cash Flow s
Years ended December 31
 
(Millions)
 
2018
 
2017
 
2016
 
Cash Flows from Operating Activities
 
 
 
 
 
 
 
Net income including noncontrolling interest
 
$
5,363 
$
4,869 
$
5,058 
Adjustments to reconcile net income including noncontrolling interest to net cash
provided by operating activities
 
 
 
 
 
 
 
Depreciation and amortization
 
 
1,488 
 
1,544 
 
1,474 
Company pension and postretirement contrib


In [6]:
print("Does the evidence contain '1,577'?", "1,577" in evidence)
print("Does the evidence contain '1,828'?", "1,828" in evidence)

Does the evidence contain '1,577'? True
Does the evidence contain '1,828'? True


In [7]:
for term in ["Purchases of property", "Purchases of marketable"]:
    pos = evidence.find(term)
    print(f"\n{evidence[pos:pos + 150]}")


Purchases of property, plant and equipment (PP&E)
 
 
(1,577) 
 
(1,373) 
 
(1,420) 
Proceeds from sale of PP&E and other assets
 
 
262 
 
49 
 
58 


Purchases of marketable securities and investments
 
 
(1,828) 
 
(2,152) 
 
(1,410) 
Proceeds from maturities and sale of marketable securities and i


In [8]:
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "qwen3:4b",
        "prompt": "Explain AI hallucination in 3 simple bullet points.",
        "stream": False
    }
)

result = response.json()
print(result["response"])

Here's a clear 3-bullet explanation of AI hallucination:

*   **AI makes up facts or stories it didn't actually learn** – like inventing false details about historical events, technical specs, or personal experiences.
*   **It's not intentional lying** but a glitch in the model's ability to accurately recall or reason from its training data – the AI "hallucinates" based on patterns it misinterprets.
*   **This breaks trust** because the AI gives answers that seem plausible but are factually incorrect, especially when used for critical tasks like research, medicine, or decisions.


In [9]:
# With Evidence

example = dataset["train"][0]

question = example["question"]
evidence = example["evidence"][0]["evidence_text"]

prompt = f"""
Answer the financial question using ONLY the evidence provided.

Question:
{question}

Evidence:
{evidence}

Instructions:
- Answer using the evidence only.
- If the evidence does not support an answer, say so.
- Do not invent facts.

Answer:
"""

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "qwen3:4b",
        "prompt": prompt,
        "stream": False
    }
)

answer = response.json()["response"]
print("QUESTION:", question)
print("\nMODEL ANSWER:", answer)
print("\nEXPECTED ANSWER:", example["answer"])

QUESTION: What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.

MODEL ANSWER: The capital expenditure amount for FY2018 is reported under "Purchases of property, plant and equipment (PP&E)" in the Cash Flows from Investing Activities section of the cash flow statement. For 2018, this line item shows a cash outflow of $1,577 million. This represents the amount spent on capital expenditures, as it directly corresponds to the purchase of long-term assets.

Thus, the FY2018 capital expenditure amount for 3M is 1577 USD millions.

EXPECTED ANSWER: $1577.00


In [10]:
# Without evidence

prompt = f"""
Answer the following financial question.

Question:
{question}

Answer:
"""

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "qwen3:4b",
        "prompt": prompt,
        "stream": False
    }
)

baseline_answer = response.json()["response"]

print("QUESTION:", question)
print("\nBASELINE ANSWER:", baseline_answer)
print("\nEXPECTED ANSWER:", example["answer"])

QUESTION: What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.

BASELINE ANSWER: Based on the cash flow statement details from **3M Company's FY2018 (fiscal year ending December 31, 2018) annual report** (specifically the 10-K filing for the fiscal year 2018), the capital expenditure (CapEx) amount is:  

**$1,826 million USD**.  

### Key detail from the cash flow statement:
- **Section**: *Cash flows from investing activities*  
- **Line item**: *Capital expenditures*  
- **Amount**: **$1,826 million** (as reported in 3M's FY2018 10-K, Page 50 of the 2018 Annual Report).  

This figure represents the cash outflow for capital expenditures during FY2018, consistent with US GAAP reporting standards. No tax adjustments or conversions were applied, as the value is reported in USD millions as per 3M's disclosures.  

*Note: FY2018 for 3M aligns with the fiscal year ending December

In [11]:
results = [{
    "financebench_id": example["financebench_id"],
    "question": question,
    "expected_answer": example["answer"],
    "baseline_answer": baseline_answer,
    "evidence_answer": answer
}]

import pandas as pd

results_df = pd.DataFrame(results)
results_df

,financebench_id,question,expected_answer,baseline_answer,evidence_answer
0,financebench_id_03029,What is the FY2018 capital expenditure amount ...,$1577.00,Based on the cash flow statement details from ...,The capital expenditure amount for FY2018 is r...


In [12]:
verification_prompt = f"""
You are a financial answer verifier.

Question:
{question}

Evidence:
{evidence}

Proposed answer:
The FY2018 capital expenditure amount was $1,826 million.

Check whether the proposed answer is supported by the evidence.

Return:
VERDICT: SUPPORTED or NOT_SUPPORTED
REASON: Explain briefly using the evidence.
"""

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "qwen3:4b",
        "prompt": verification_prompt,
        "stream": False
    }
)

verification_result = response.json()["response"]
print(verification_result)

VERDICT: NOT_SUPPORTED  
REASON: The evidence shows "Purchases of property, plant and equipment (PP&E)" for FY2018 was $1,577 million (a negative value in the cash flow statement indicating outflow). The proposed answer of $1,826 million does not match this line item. The $1,826 figure is close to the "Purchases of marketable securities and investments" ($1,828 million), but this represents debt/investment activity, not capital expenditure (which specifically refers to PP&E purchases). Capital expenditure is explicitly defined by the "Purchases of property, plant and equipment" line in the cash flow statement, not marketable securities.


In [15]:
from collections import Counter

records = dataset["train"]

print("Total questions:", len(records))
print("\nQuestion types:")
print(Counter(row["question_type"] for row in records))

print("\nDocument types:")
print(Counter(row["doc_type"] for row in records))

print("\nQuestions with multiple evidence passages:",
      sum(len(row["evidence"]) > 1 for row in records))

Total questions: 150

Question types:
Counter({'metrics-generated': 50, 'domain-relevant': 50, 'novel-generated': 50})

Document types:
Counter({'10k': 112, '10q': 15, 'Earnings': 14, '8k': 9})

Questions with multiple evidence passages: 35


In [16]:

import requests
import json
import time
from pathlib import Path

records = dataset["train"]
output_file = Path("baseline_results.jsonl")

def ask_qwen(prompt):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "qwen3:4b",
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0}
        },
        timeout=300
    )
    response.raise_for_status()
    return response.json()["response"]

# Start fresh for this run
completed = 0

with output_file.open("w", encoding="utf-8") as f:
    for i, example in enumerate(records):
        question = example["question"]

        # Combine every evidence passage for this question
        evidence = "\n\n".join(
            item["evidence_text"]
            for item in example["evidence"]
        )

        # Experiment A: no evidence
        baseline = ask_qwen(
            f"Answer this financial question concisely.\n"
            f"Question: {question}\nAnswer:"
        )

        # Experiment B: evidence provided
        evidence_answer = ask_qwen(
            "Answer the financial question using only the "
            "provided evidence. If the evidence is insufficient, "
            "say so. Do not invent facts.\n\n"
            f"Question: {question}\n\n"
            f"Evidence:\n{evidence}\n\nAnswer:"
        )

        result = {
            "financebench_id": example["financebench_id"],
            "question_type": example["question_type"],
            "question": question,
            "expected_answer": example["answer"],
            "evidence": evidence,
            "baseline_answer": baseline,
            "evidence_answer": evidence_answer
        }

        f.write(json.dumps(result, ensure_ascii=False) + "\n")
        f.flush()

        completed += 1
        print(f"Completed {completed}/{len(records)}")

print(f"\nSaved {completed} questions to {output_file}")


Completed 1/150
Completed 2/150
Completed 3/150
Completed 4/150
Completed 5/150
Completed 6/150
Completed 7/150
Completed 8/150
Completed 9/150
Completed 10/150
Completed 11/150
Completed 12/150
Completed 13/150
Completed 14/150
Completed 15/150
Completed 16/150
Completed 17/150
Completed 18/150
Completed 19/150
Completed 20/150
Completed 21/150
Completed 22/150
Completed 23/150
Completed 24/150
Completed 25/150
Completed 26/150
Completed 27/150
Completed 28/150
Completed 29/150
Completed 30/150
Completed 31/150
Completed 32/150
Completed 33/150
Completed 34/150
Completed 35/150
Completed 36/150
Completed 37/150
Completed 38/150
Completed 39/150
Completed 40/150
Completed 41/150
Completed 42/150
Completed 43/150
Completed 44/150
Completed 45/150
Completed 46/150
Completed 47/150
Completed 48/150
Completed 49/150
Completed 50/150
Completed 51/150
Completed 52/150
Completed 53/150
Completed 54/150
Completed 55/150
Completed 56/150
Completed 57/150
Completed 58/150
Completed 59/150
Comple

In [17]:
import pandas as pd

df = pd.read_json("baseline_results.jsonl", lines=True)

print("Total questions:", len(df))
print("Missing baseline answers:", df["baseline_answer"].isna().sum())
print("Missing evidence answers:", df["evidence_answer"].isna().sum())
print("Duplicate question IDs:", df["financebench_id"].duplicated().sum())

Total questions: 150
Missing baseline answers: 0
Missing evidence answers: 0
Duplicate question IDs: 0


In [18]:
df[[
    "question",
    "expected_answer",
    "baseline_answer",
    "evidence_answer"
]].head(5)

,question,expected_answer,baseline_answer,evidence_answer
0,What is the FY2018 capital expenditure amount ...,$1577.00,"10,500 (USD millions) \n*Source: 3M Company F...",Based solely on the provided cash flow stateme...
1,Assume that you are a public equities analyst....,$8.70,10.0,Based solely on the provided balance sheet evi...
2,Is 3M a capital-intensive business based on FY...,"No, the company is managing its CAPEX and Fixe...",Yes. 3M's FY2022 CapEx/revenue ratio was **~29...,Based solely on the provided evidence for FY20...
3,What drove operating margin change as of FY202...,Operating Margin for 3M in FY2022 has decrease...,For 3M FY2022: \n**Operating margin decreased...,The evidence provided is insufficient to answe...
4,"If we exclude the impact of M&A, which segment...",The consumer segment shrunk by 0.9% organically.,Other segment. (3M reported negative growth of...,"Based solely on the provided evidence, the seg..."


In [9]:

def ask_qwen(prompt):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "qwen3:4b",
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0}
        },
        timeout=300
    )
    response.raise_for_status()
    return response.json()["response"]


In [11]:
import json, re, requests
from pathlib import Path

records = dataset["train"]
output_file = Path("results_available.jsonl")

PROMPT = """You are a financial analyst. Answer the question using ONLY the evidence below.

Question:
{question}

Evidence:
{evidence}

Rules:
- Use only the evidence. Do not use outside knowledge.
- If the evidence does not contain enough information to answer, your final answer must be: INSUFFICIENT_EVIDENCE
- Keep your explanation short (at most 5 lines).
- The LAST line of your reply must be exactly in this format:
FINAL ANSWER: <your short answer>
"""

def clean(text):
    # remove Qwen's hidden "thinking" section if it appears
    return re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()

def get_final_answer(text):
    matches = re.findall(r"FINAL ANSWER:\s*(.+)", text)
    return matches[-1].strip() if matches else None

# Resume support: skip questions already done if the run gets interrupted
done = set()
if output_file.exists():
    with output_file.open() as f:
        done = {json.loads(line)["financebench_id"] for line in f}

with output_file.open("a", encoding="utf-8") as f:
    for i, ex in enumerate(records):
        if ex["financebench_id"] in done:
            continue
        evidence = "\n\n".join(item["evidence_text"] for item in ex["evidence"])
        raw = clean(ask_qwen(PROMPT.format(question=ex["question"], evidence=evidence)))
        row = {
            "financebench_id": ex["financebench_id"],
            "question_type": ex["question_type"],
            "condition": "available",
            "question": ex["question"],
            "expected_answer": ex["answer"],
            "evidence": evidence,
            "raw_answer": raw,
            "final_answer": get_final_answer(raw),
        }
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
        f.flush()
        print(f"{i+1}/150  ->  {row['final_answer']}")

1/150  ->  1577
2/150  ->  8.74
3/150  ->  Yes
4/150  ->  INSUFFICIENT_EVIDENCE
5/150  ->  Consumer
6/150  ->  No
7/150  ->  1.500% Notes due 2026, 1.750% Notes due 2030, and 1.500% Notes due 2031
8/150  ->  Yes
9/150  ->  24.25
10/150  ->  1.9
11/150  ->  0.66
12/150  ->  65.4
13/150  ->  0.83
14/150  ->  No
15/150  ->  Yes
16/150  ->  0
17/150  ->  12
18/150  ->  -0.01
19/150  ->  93.76
20/150  ->  30.8
21/150  ->  11588
22/150  ->  1615.9
23/150  ->  INSUFFICIENT_EVIDENCE
24/150  ->  improved
25/150  ->  INSUFFICIENT_EVIDENCE
26/150  ->  packaging industry
27/150  ->  No
28/150  ->  INSUFFICIENT_EVIDENCE
29/150  ->  2018
30/150  ->  None
31/150  ->  4.19%
32/150  ->  Yes
33/150  ->  server CPUs/GPUs, data center products (DPUs, FPGAs, Adaptive SoCs), PC components (CPUs, APUs, chipsets), discrete GPUs, and embedded SoCs
34/150  ->  64% increase in Data Center segment revenue from EPYC server processors, 21% increase in Gaming segment revenue from semi-custom products, and significan